# HMM POS Tagging

In [ ]:
# ============================================================
# 05 - POS TAGGING USING HMMLEARN
# ============================================================

# Install once if required:
# !pip install hmmlearn

import numpy as np

try:
    from hmmlearn.hmm import CategoricalHMM
except ImportError:
    # Older hmmlearn versions may expose MultinomialHMM.
    # For this lab template, install a recent hmmlearn version.
    raise ImportError("Install/update hmmlearn so CategoricalHMM is available.")

# ============================================================
# STATES = POS TAGS
# ============================================================

states = ["NN", "VB", "RB"]
state2idx = {state: i for i, state in enumerate(states)}

# ============================================================
# SMALL MANUALLY SPECIFIED VOCABULARY
# ============================================================

vocab = [
    "<UNK>",
    "dogs",
    "cats",
    "run",
    "chase",
    "quickly",
    "slowly"
]

word2idx = {word: i for i, word in enumerate(vocab)}

# ============================================================
# START PROBABILITIES
# P(first POS tag)
# ============================================================

startprob = np.array([
    0.7,  # NN
    0.2,  # VB
    0.1   # RB
])

# ============================================================
# TRANSITION PROBABILITIES
# P(current tag | previous tag)
# Rows = previous state
# Columns = current state
# ============================================================

transmat = np.array([
    [0.1, 0.7, 0.2],  # NN -> NN, VB, RB
    [0.6, 0.1, 0.3],  # VB -> NN, VB, RB
    [0.5, 0.2, 0.3],  # RB -> NN, VB, RB
])

# ============================================================
# EMISSION PROBABILITIES
# P(word | POS tag)
# Rows = POS states
# Columns = vocabulary
# ============================================================

emissionprob = np.array([
    # <UNK> dogs cats run chase quickly slowly
    [0.02, 0.35, 0.35, 0.03, 0.05, 0.10, 0.10],  # NN
    [0.02, 0.03, 0.03, 0.40, 0.40, 0.10, 0.02],  # VB
    [0.02, 0.02, 0.02, 0.02, 0.02, 0.45, 0.45],  # RB
])

# ============================================================
# CREATE CATEGORICAL HMM
# ============================================================

model = CategoricalHMM(
    n_components=len(states),
    init_params=""
)

model.startprob_ = startprob
model.transmat_ = transmat
model.emissionprob_ = emissionprob

# ============================================================
# TAG A SENTENCE
# ============================================================

def hmm_tag_sentence(words):
    # Unknown words become <UNK>
    obs = [
        word2idx.get(word.lower(), word2idx["<UNK>"])
        for word in words
    ]

    X = np.array(obs).reshape(-1, 1)

    log_probability, state_sequence = model.decode(
        X,
        algorithm="viterbi"
    )

    tags = [states[i] for i in state_sequence]

    return list(zip(words, tags)), log_probability


test_sentence = ["dogs", "run", "quickly"]

result, log_probability = hmm_tag_sentence(test_sentence)

print("Input:")
print(test_sentence)

print("\nPOS tags:")
print(result)

print("\nViterbi log probability:")
print(log_probability)

# ============================================================
# WHAT EACH HMM PART MEANS
# ============================================================
# states          -> hidden POS tags
# observations    -> words
# startprob_      -> probability of first POS tag
# transmat_       -> transition probabilities between tags
# emissionprob_   -> probability of a word given a tag
# decode(...viterbi)
#                 -> finds most likely hidden POS sequence

# ============================================================
# ============================================================
# states
# vocab
# startprob
# transmat
# emissionprob
# test_sentence
#
# If a test word is not in vocab, it is automatically mapped
# to <UNK>, so the code does not crash.


Input:
['dogs', 'run', 'quickly']

POS tags:
[('dogs', 'NN'), ('run', 'VB'), ('quickly', 'RB')]

Viterbi log probability:
-5.8...
